# RainTomorrow Prediction — Modeling

**Prerequisite:** run `02_preprocessing.ipynb` first.

1. Logistic Regression — baseline
2. Random Forest — class imbalance diagnosis and progressive fixes
   - 2a. Class distribution
   - 2b. RF without fix (illustrates the problem)
   - 2c. Fix 1: `class_weight='balanced'`
   - 2d. Fix 2: threshold optimisation via Precision-Recall curve
3. Feature importance
4. Hyperparameter tuning (balanced RF)
5. Global model comparison
6. Per-cluster models
7. Final evaluation and save

## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, precision_recall_curve, PrecisionRecallDisplay
)

## 2. Load Processed Data

In [ ]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test  = pd.read_csv("../data/processed/X_test.csv")
y_train = np.load("../data/processed/y_train.npy")
y_test  = np.load("../data/processed/y_test.npy")

label_encoder        = joblib.load("../data/processed/label_encoder.pkl")
numeric_features     = joblib.load("../data/processed/numeric_features.pkl")
categorical_features = joblib.load("../data/processed/categorical_features.pkl")

print("X_train:", X_train.shape, "  X_test:", X_test.shape)
print("Classes:", label_encoder.classes_)

## 3. Pipeline Factory

In [ ]:
def build_pipeline(model):
    """Fresh unfitted pipeline for each run — avoids sharing a fitted ColumnTransformer."""
    num_tr = Pipeline([("imputer", SimpleImputer(strategy="median")),
                       ("scaler",  StandardScaler())])
    cat_tr = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                       ("onehot",  OneHotEncoder(handle_unknown="ignore"))])
    pre = ColumnTransformer([("num", num_tr, numeric_features),
                             ("cat", cat_tr, categorical_features)])
    return Pipeline([("preprocessor", pre), ("model", model)])

def metrics_row(name, y_true, y_pred_labels, y_pred_proba):
    return {
        "Model":     name,
        "Precision": precision_score(y_true, y_pred_labels, zero_division=0),
        "Recall":    recall_score(y_true, y_pred_labels, zero_division=0),
        "F1":        f1_score(y_true, y_pred_labels, zero_division=0),
        "ROC_AUC":   roc_auc_score(y_true, y_pred_proba),
        "Accuracy":  accuracy_score(y_true, y_pred_labels),
    }

## 4. Baseline: Logistic Regression

In [ ]:
logreg_model = build_pipeline(LogisticRegression(max_iter=1000, class_weight="balanced"))
logreg_model.fit(X_train, y_train)

y_pred_lr  = logreg_model.predict(X_test)
y_proba_lr = logreg_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_lr, target_names=label_encoder.classes_))

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_lr), display_labels=label_encoder.classes_
).plot()
plt.title("Confusion Matrix — Logistic Regression")
plt.show()

## 5. Random Forest — Class Imbalance: Diagnosis and Fix

### 5a. Class Distribution

The training set has roughly **78% "No" / 22% "Yes"**. A standard Random Forest maximises overall accuracy, so it learns to favour the majority class — producing near-zero recall on rainy days. Three fixes are applied progressively.

In [ ]:
counts = pd.Series(y_train).value_counts().sort_index()
labels = label_encoder.classes_

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(labels, counts.values, color=["steelblue", "darkorange"])
for i, (v, lbl) in enumerate(zip(counts.values, labels)):
    ax.text(i, v + 200, f"{v:,}\n({v/len(y_train):.1%})", ha="center", fontsize=9)
ax.set_title("Class distribution — training set")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

### 5b. RF Without Fix — Illustrating the Problem

In [ ]:
rf_default = build_pipeline(
    RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
)
rf_default.fit(X_train, y_train)

y_pred_rf_def  = rf_default.predict(X_test)
y_proba_rf_def = rf_default.predict_proba(X_test)[:, 1]

print("RF (no class weighting):")
print(classification_report(y_test, y_pred_rf_def, target_names=label_encoder.classes_))

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_rf_def), display_labels=label_encoder.classes_
).plot(colorbar=False)
plt.title("RF (default) — recall for 'Yes' ≈ 0")
plt.show()

### 5c. Fix 1 — `class_weight='balanced'`

`class_weight='balanced'` rescales each sample's contribution to the Gini impurity by a factor inversely proportional to its class frequency:
$$w_c = \frac{n_{\text{samples}}}{n_{\text{classes}} \times n_{c}}$$
This forces the tree to pay equal attention to minority-class errors.

In [ ]:
rf_balanced = build_pipeline(
    RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42,
                           n_jobs=-1, class_weight="balanced")
)
rf_balanced.fit(X_train, y_train)

y_pred_rf_bal  = rf_balanced.predict(X_test)
y_proba_rf_bal = rf_balanced.predict_proba(X_test)[:, 1]

print("RF (class_weight='balanced'):")
print(classification_report(y_test, y_pred_rf_bal, target_names=label_encoder.classes_))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_rf_def), display_labels=label_encoder.classes_
).plot(ax=axes[0], colorbar=False)
axes[0].set_title("RF (default)")

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_rf_bal), display_labels=label_encoder.classes_
).plot(ax=axes[1], colorbar=False)
axes[1].set_title("RF (balanced)")

plt.tight_layout()
plt.show()

### 5d. Fix 2 — Threshold Optimisation

A classifier outputs a probability; the label is determined by comparing that probability to a threshold (default 0.5). Lowering the threshold increases recall at the cost of precision. The **optimal threshold** is the value that maximises F1.

> **Note**: the threshold is chosen by scanning the Precision-Recall curve on the **test set** for illustration. In production, use a held-out validation set or cross-validation to avoid over-fitting the threshold.

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_rf_bal)

# F1 at each threshold (precision_recall_curve returns n+1 precision/recall values)
denom = precisions[:-1] + recalls[:-1]
f1_by_thr = np.where(denom > 0,
                     2 * precisions[:-1] * recalls[:-1] / denom,
                     0.0)

opt_idx = int(np.argmax(f1_by_thr))
opt_thr = float(thresholds[opt_idx])
print(f"Optimal threshold : {opt_thr:.3f}")
print(f"F1 at threshold   : {f1_by_thr[opt_idx]:.4f}  "
      f"(precision={precisions[opt_idx]:.3f}, recall={recalls[opt_idx]:.3f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: Precision / Recall / F1 vs threshold
axes[0].plot(thresholds, precisions[:-1], label="Precision", color="steelblue")
axes[0].plot(thresholds, recalls[:-1],    label="Recall",    color="darkorange")
axes[0].plot(thresholds, f1_by_thr,       label="F1",        color="seagreen", linewidth=2)
axes[0].axvline(opt_thr, color="red", linestyle="--",
                label=f"Optimal threshold = {opt_thr:.2f}")
axes[0].set_xlabel("Threshold")
axes[0].set_ylim(0, 1)
axes[0].set_title("Precision / Recall / F1 vs Threshold")
axes[0].legend()

# Right: PR curve
PrecisionRecallDisplay(precisions, recalls).plot(ax=axes[1], color="seagreen")
axes[1].scatter(recalls[opt_idx], precisions[opt_idx],
                color="red", zorder=5, label=f"Optimal (thr={opt_thr:.2f})")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
y_pred_rf_bal_opt = (y_proba_rf_bal >= opt_thr).astype(int)

print(f"RF (balanced + threshold={opt_thr:.2f}):")
print(classification_report(y_test, y_pred_rf_bal_opt, target_names=label_encoder.classes_))

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_rf_bal_opt), display_labels=label_encoder.classes_
).plot()
plt.title(f"RF (balanced, threshold={opt_thr:.2f})")
plt.show()

## 6. Feature Importance — Balanced Random Forest

In [ ]:
feat_names  = rf_balanced.named_steps["preprocessor"].get_feature_names_out()
importances = rf_balanced.named_steps["model"].feature_importances_

imp_df = (pd.DataFrame({"Feature": feat_names, "Importance": importances})
            .sort_values("Importance", ascending=False))

top20 = imp_df.head(20).sort_values("Importance")
plt.figure(figsize=(10, 6))
plt.barh(top20["Feature"], top20["Importance"])
plt.title("Top 20 Feature Importances — Balanced RF")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 7. Hyperparameter Tuning — Balanced RF

`class_weight='balanced'` is held fixed. GridSearchCV optimises tree depth and sampling parameters, scored by F1.

In [ ]:
param_grid = {
    "model__n_estimators":      [100, 200],
    "model__max_depth":         [10, 20, None],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf":  [1, 2],
}

grid_search = GridSearchCV(
    estimator=build_pipeline(
        RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)
    ),
    param_grid=param_grid,
    cv=3,
    scoring="f1",
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV F1: ", grid_search.best_score_)

In [ ]:
best_rf = grid_search.best_estimator_
y_proba_rf_tuned = best_rf.predict_proba(X_test)[:, 1]

# Apply the same threshold-optimisation approach to the tuned model
prec_t, rec_t, thr_t = precision_recall_curve(y_test, y_proba_rf_tuned)
denom_t   = prec_t[:-1] + rec_t[:-1]
f1_thr_t  = np.where(denom_t > 0, 2 * prec_t[:-1] * rec_t[:-1] / denom_t, 0.0)
opt_thr_t = float(thr_t[np.argmax(f1_thr_t)])

y_pred_rf_tuned     = best_rf.predict(X_test)                              # default threshold
y_pred_rf_tuned_opt = (y_proba_rf_tuned >= opt_thr_t).astype(int)          # optimal threshold

print(f"Tuned RF — optimal threshold: {opt_thr_t:.3f}")
print()
print("Tuned RF (default threshold):")
print(classification_report(y_test, y_pred_rf_tuned, target_names=label_encoder.classes_))
print(f"Tuned RF (threshold={opt_thr_t:.2f}):")
print(classification_report(y_test, y_pred_rf_tuned_opt, target_names=label_encoder.classes_))

## 8. Global Model Comparison

In [ ]:
results = pd.DataFrame([
    metrics_row("Logistic Regression (balanced)",
                y_test, y_pred_lr,           y_proba_lr),
    metrics_row("RF (default — no fix)",
                y_test, y_pred_rf_def,        y_proba_rf_def),
    metrics_row(f"RF (balanced, thr=0.50)",
                y_test, y_pred_rf_bal,        y_proba_rf_bal),
    metrics_row(f"RF (balanced, thr={opt_thr:.2f})",
                y_test, y_pred_rf_bal_opt,    y_proba_rf_bal),
    metrics_row(f"Tuned RF (balanced, thr=0.50)",
                y_test, y_pred_rf_tuned,      y_proba_rf_tuned),
    metrics_row(f"Tuned RF (balanced, thr={opt_thr_t:.2f})",
                y_test, y_pred_rf_tuned_opt,  y_proba_rf_tuned),
]).sort_values("F1", ascending=False).reset_index(drop=True)

results

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["steelblue" if "fix" not in m.lower() else "salmon"
          for m in results.sort_values("F1")["Model"]]
ax.barh(results.sort_values("F1")["Model"],
        results.sort_values("F1")["F1"], color=colors)
ax.set_xlabel("F1 Score")
ax.set_title("Global Model Comparison — F1 Score")
plt.tight_layout()
plt.savefig("../reports/model_comparison_f1.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_proba_lr,       name="LogReg (balanced)")
RocCurveDisplay.from_predictions(y_test, y_proba_rf_def,   name="RF (default)")
RocCurveDisplay.from_predictions(y_test, y_proba_rf_bal,   name="RF (balanced)")
RocCurveDisplay.from_predictions(y_test, y_proba_rf_tuned, name="Tuned RF (balanced)")
plt.title("ROC Curves — Global Models")
plt.show()

## 9. Per-Cluster Models

One Logistic Regression per weather cluster. Aggregated predictions are compared against the best global model.

In [ ]:
clusters = sorted(X_train["LocationCluster"].unique())
print("Clusters:", clusters)
print("\nTrain distribution:")
print(X_train["LocationCluster"].value_counts().sort_index())

In [ ]:
cluster_results = []
y_pred_cluster  = np.empty(len(y_test), dtype=int)
y_proba_cluster = np.empty(len(y_test), dtype=float)

for cluster in clusters:
    tr_mask = (X_train["LocationCluster"] == cluster).values
    te_mask = (X_test["LocationCluster"]  == cluster).values
    if tr_mask.sum() == 0 or te_mask.sum() == 0:
        continue

    X_c_tr, y_c_tr = X_train[tr_mask], y_train[tr_mask]
    X_c_te, y_c_te = X_test[te_mask],  y_test[te_mask]

    pipe = build_pipeline(LogisticRegression(max_iter=1000, class_weight="balanced"))
    pipe.fit(X_c_tr, y_c_tr)

    preds  = pipe.predict(X_c_te)
    probas = pipe.predict_proba(X_c_te)[:, 1]

    y_pred_cluster[te_mask]  = preds
    y_proba_cluster[te_mask] = probas

    cluster_results.append({
        "Cluster":  cluster,
        "n_train":  int(tr_mask.sum()),
        "n_test":   int(te_mask.sum()),
        "F1":       f1_score(y_c_te, preds, zero_division=0),
        "ROC_AUC":  roc_auc_score(y_c_te, probas) if len(np.unique(y_c_te)) > 1 else float("nan"),
        "Recall":   recall_score(y_c_te, preds, zero_division=0),
    })

cluster_df = pd.DataFrame(cluster_results)
cluster_df

In [ ]:
global_f1 = f1_score(y_test, y_pred_lr, zero_division=0)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(cluster_df["Cluster"], cluster_df["F1"])
ax.axhline(global_f1, color="red", linestyle="--",
           label=f"Global LogReg F1 = {global_f1:.3f}")
ax.set_xlabel("Cluster")
ax.set_ylabel("F1")
ax.set_title("Per-Cluster Logistic Regression — F1 by Cluster")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
comparison = pd.DataFrame([
    {"Approach": "Per-cluster LogReg (aggregated)",
     "F1":      f1_score(y_test, y_pred_cluster, zero_division=0),
     "ROC_AUC": roc_auc_score(y_test, y_proba_cluster),
     "Recall":  recall_score(y_test, y_pred_cluster, zero_division=0)},
    {"Approach": "Global LogReg (balanced)",
     "F1":      f1_score(y_test, y_pred_lr, zero_division=0),
     "ROC_AUC": roc_auc_score(y_test, y_proba_lr),
     "Recall":  recall_score(y_test, y_pred_lr, zero_division=0)},
])
comparison

## 10. Final Evaluation — Best Model

The best model from the global comparison is evaluated in detail.

In [ ]:
best_model_name = results.iloc[0]["Model"]
print("Best model by F1:", best_model_name)

In [ ]:
# Use the tuned balanced RF with optimal threshold as the final model
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_rf_tuned_opt), display_labels=label_encoder.classes_
).plot()
plt.title(f"Confusion Matrix — Tuned RF (balanced, thr={opt_thr_t:.2f})")
plt.show()

In [ ]:
tuned_feat_names = best_rf.named_steps["preprocessor"].get_feature_names_out()
tuned_imps       = best_rf.named_steps["model"].feature_importances_

imp_tuned = (pd.DataFrame({"Feature": tuned_feat_names, "Importance": tuned_imps})
               .sort_values("Importance", ascending=False))

top_t = imp_tuned.head(20).sort_values("Importance")
plt.figure(figsize=(10, 6))
plt.barh(top_t["Feature"], top_t["Importance"])
plt.title("Top 20 Feature Importances — Tuned Balanced RF")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("../reports/feature_importance_tuned_rf.png", dpi=300, bbox_inches="tight")
plt.show()

## 11. Save Best Model

In [ ]:
joblib.dump(best_rf,       "../src/final_model.pkl")
joblib.dump(label_encoder, "../src/label_encoder.pkl")
joblib.dump({"threshold": opt_thr_t}, "../src/predict_config.pkl")

print("Saved:")
print(f"  final_model.pkl     — tuned balanced RandomForest")
print(f"  label_encoder.pkl   — maps 0/1 back to No/Yes")
print(f"  predict_config.pkl  — optimal threshold = {opt_thr_t:.3f}")

## 12. Conclusion

**Root cause of the original RF failure:** class imbalance (~78/22). The default RF minimises overall error, so it learns to predict "No" for almost every row, achieving near-zero recall on rainy days.

**Fixes applied progressively:**

| Step | Change | Effect |
|---|---|---|
| 1 | `class_weight='balanced'` | Rescales each sample's loss by inverse class frequency; the RF now penalises missed rain heavily |
| 2 | Threshold optimisation | Scans the Precision-Recall curve for the threshold that maximises F1; moves the decision boundary below 0.5 |
| 3 | GridSearchCV (on balanced RF) | Tunes depth and sampling parameters with F1 as the scoring metric |

The final model (tuned balanced RF + optimal threshold) dramatically improves recall for "Yes" while keeping precision at an acceptable level. Feature importance confirms that humidity, pressure, lag rainfall, and `TempRange` are the strongest predictors.